# LAB 3 — Métodos Supervisados para Regresión

Estructura equivalente a los laboratorios anteriores: carga/EDA, preparación, configuración estándar, variaciones, benchmark, selección del mejor modelo y análisis final.


## 1. Librerías e importación del paquete


In [ ]:
from pathlib import Path
import pandas as pd

from ModuloRegresionLab03 import (
    cargar_csv_o_diabetes, resumen_dataset, detectar_problema_regresion,
    graficar_eda_basica, ExperimentoRegresion
)


## 2. Carga del dataset

Por defecto se usa el dataset `diabetes_regression_lab03.csv` incluido. Si el profesor brinda otro CSV, cambie `DATASET_PATH` y `COLUMNA_OBJETIVO`.


In [ ]:
DATASET_PATH = 'diabetes_regression_lab03.csv'
COLUMNA_OBJETIVO = 'target'
NOMBRE_DATASET = 'Diabetes'

df, target = cargar_csv_o_diabetes(DATASET_PATH, COLUMNA_OBJETIVO)
df.head()


In [ ]:
resumen = resumen_dataset(df, target)
pd.DataFrame([resumen])


In [ ]:
if not detectar_problema_regresion(df, target):
    print('Advertencia: la variable objetivo no parece continua; revise si el problema corresponde a clasificación.')
else:
    print('La variable objetivo parece continua. El dataset es adecuado para regresión.')


## 3. EDA básico


In [ ]:
CARPETA_FIGURAS = Path('outputs_lab03/figures')
CARPETA_RESULTADOS = Path('outputs_lab03/results')
CARPETA_FIGURAS.mkdir(parents=True, exist_ok=True)
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)

graficar_eda_basica(df, target, str(CARPETA_FIGURAS))
df.describe().T


## 4. Experimento de regresión

Modelos incluidos: LinearRegression, Ridge, RidgeCV, Lasso, LassoCV, KNN, SVR, DecisionTreeRegressor, RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor y XGBoostRegressor si está instalado.


In [ ]:
exp = ExperimentoRegresion(
    df=df,
    columna_objetivo=target,
    nombre_dataset=NOMBRE_DATASET,
    carpeta='outputs_lab03',
    test_size=0.25,
    random_state=42,
    cv_splits=3
)
exp.ejecutar(verbose=True)


## 5. Benchmark general


In [ ]:
benchmark = exp.benchmark()
benchmark.round(4)


In [ ]:
mejores = exp.mejores_por_algoritmo()
mejores.round(4)


## 6. Guardado de resultados y gráficos


In [ ]:
exp.guardar_resultados(str(CARPETA_RESULTADOS))
exp.plot_benchmark_rmse(str(CARPETA_FIGURAS / 'benchmark_rmse.png'))
exp.plot_benchmark_r2(str(CARPETA_FIGURAS / 'benchmark_r2.png'))
exp.plot_familias(str(CARPETA_FIGURAS / 'benchmark_familias.png'))
exp.analisis_mejor(str(CARPETA_FIGURAS))


## 7. Interpretación del mejor modelo


In [ ]:
mejor = exp.mejor_modelo
print('Mejor modelo:', exp.mejor_nombre)
pd.DataFrame([mejor.metricas_test]).round(4)


In [ ]:
top_vars = mejor.top_variables(15)
top_vars.round(4) if top_vars is not None else 'No disponible'


## 8. Observaciones para redactar el informe

- Compare la configuración estándar contra las variaciones.
- Use RMSE como criterio principal de error y R² como criterio de capacidad explicativa.
- Interprete si la selección automática por validación cruzada coincide o no con el mejor RMSE de prueba.
- Revise los residuos para detectar sesgo, dispersión no constante o valores extremos.
